<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_2_MLR/17_2_4_5_notes_on_trees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notes on Decision Trees

This notebook provides a conceptual introduction to **decision trees** using a familiar game as an analogy.

**What this is:** A conceptual review notebook that explains how decision trees work through the "20 questions" game.

**Note:** This notebook uses a simple pedagogical example (guessing a number) rather than the Ames Housing dataset to clearly demonstrate the core concept.

## The 20 Questions Analogy

A decision tree is essentially a structured way of asking a series of yes/no questions to arrive at a prediction. This is exactly like the game "20 Questions" where one person thinks of an object, and the other asks yes/no questions to guess what it is.

### Guessing a Number Game

Imagine you're trying to guess a secret number between 1 and 100. The optimal strategy is to **split the remaining possibilities in half** at each step:

1. **First question:** "Is the number > 50?"
2. **If yes:** Ask "Is it > 75?"
3. **If no:** Ask "Is it > 25?"

This **binary split** is the foundation of decision trees!

In [1]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def draw_decision_tree():
    """Draw a simple decision tree for the number guessing game."""
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 10)
    ax.axis('off')
    ax.set_title("Decision Tree: Guessing a Number Between 1-100", fontsize=16, fontweight='bold', pad=20)

    def draw_node(x, y, text, color='lightblue'):
        """Draw a rectangular node."""
        rect = mpatches.FancyBboxPatch((x-0.6, y-0.4), 1.2, 0.8, 
                                        boxstyle="round,pad=0.05",
                                        facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        ax.text(x, y, text, ha='center', va='center', fontsize=11, fontweight='bold')
    
    def draw_edge(x1, y1, x2, y2, label):
        """Draw an edge between nodes with a label."""
        ax.annotate('', xy=(x2, y2+0.4), xytext=(x1, y1-0.4),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
        mid_x = (x1 + x2) / 2
        mid_y = (y1 + y2) / 2
        ax.text(mid_x, mid_y, label, fontsize=9, ha='center', va='center', 
                bbox=dict(boxstyle='round', facecolor='white', edgecolor='none', alpha=0.8))

    level_1_y = 8
    level_2_y = 6
    level_3_y = 4
    level_4_y = 2
    
    draw_node(7, level_1_y, "Is > 50?\n(50)", 'lightblue')
    
    draw_node(3.5, level_2_y, "Is > 25?\n(25)", 'lightgreen')
    draw_node(10.5, level_2_y, "Is > 75?\n(75)", 'lightgreen')
    
    draw_edge(7, level_1_y-0.4, 3.5, level_2_y+0.4, 'Yes')
    draw_edge(7, level_1_y-0.4, 10.5, level_2_y+0.4, 'No')
    
    draw_node(1.75, level_3_y, "Is > 12?\n(12)", 'lightyellow')
    draw_node(5.25, level_3_y, "Is > 37?\n(37)", 'lightyellow')
    draw_node(8.25, level_3_y, "Is > 62?\n(62)", 'lightyellow')
    draw_node(12.75, level_3_y, "Is > 87?\n(87)", 'lightyellow')
    
    draw_edge(3.5, level_2_y-0.4, 1.75, level_3_y+0.4, 'Yes')
    draw_edge(3.5, level_2_y-0.4, 5.25, level_3_y+0.4, 'No')
    draw_edge(10.5, level_2_y-0.4, 8.25, level_3_y+0.4, 'Yes')
    draw_edge(10.5, level_2_y-0.4, 12.75, level_3_y+0.4, 'No')
    
    draw_node(0.875, level_4_y, "Leaf\n(1-12)", 'lightcoral')
    draw_node(2.625, level_4_y, "Leaf\n(13-25)", 'lightcoral')
    draw_node(4.375, level_4_y, "Leaf\n(26-37)", 'lightcoral')
    draw_node(6.125, level_4_y, "Leaf\n(38-50)", 'lightcoral')
    draw_node(7.875, level_4_y, "Leaf\n(51-62)", 'lightcoral')
    draw_node(9.625, level_4_y, "Leaf\n(63-75)", 'lightcoral')
    draw_node(11.375, level_4_y, "Leaf\n(76-87)", 'lightcoral')
    draw_node(13.125, level_4_y, "Leaf\n(88-100)", 'lightcoral')
    
    for x in [1.75, 5.25, 8.25, 12.75]:
        draw_edge(x, level_3_y-0.4, x-0.875, level_4_y+0.4, 'Yes')
        draw_edge(x, level_3_y-0.4, x+0.875, level_4_y+0.4, 'No')
    
    ax.text(7, 0.3, "Each path from root to leaf represents a sequence of yes/no answers", 
            ha='center', fontsize=10, style='italic', color='gray')
    
    plt.tight_layout()
    plt.show()

draw_decision_tree()

## Key Decision Tree Terminology

| Term | Definition | In Our Game |
|-----|-------------|--------------|
| **Root** | The starting node | "Is > 50?" |
| **Split** | A decision point | Each yes/no question |
| **Branch** | A path from one node to another | The "Yes" or "No" answer |
| **Leaf** | A terminal node with no children | The final guess range |
| **Depth** | Number of levels in the tree | How many questions asked |
| **Impurity** | How mixed the groups are at a node | How close the range is to one answer |

## How This Applies to Real Data

In the number guessing game, we split exactly in half at each step. In real ML problems:

- **Classification:** Split based on features that best separate the classes (e.g., "Is price > $200K?")
- **Regression:** Split based on features that reduce the variance in the target variable

The algorithm (like CART - Classification and Regression Trees) automatically finds the **optimal splits** that maximize information gain or minimize impurity.

In [2]:
def draw_simple_vs_complex():
    """Compare underfit, optimal, and overfit trees."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    titles = ['Underfit (Too Simple)', 'Optimal (Balanced)', 'Overfit (Too Complex)']
    colors = ['lightblue', 'lightblue', 'lightblue']
    
    for idx, ax in enumerate(axes):
        ax.set_xlim(0, 10)
        ax.set_ylim(0, 6)
        ax.axis('off')
        ax.set_title(titles[idx], fontsize=12, fontweight='bold')
        
        if idx == 0:
            ax.text(5, 5.5, "Is > 50?", ha='center', va='center', fontsize=14, 
                    bbox=dict(boxstyle='round', facecolor='lightblue', edgecolor='black'))
            ax.text(5, 3, "PREDICT:\nAverage", ha='center', va='center', fontsize=11,
                    bbox=dict(boxstyle='round', facecolor='lightcoral', edgecolor='black'))
            ax.annotate('', xy=(5, 4), xytext=(5, 5), arrowprops=dict(arrowstyle='->'))
            ax.text(5, 1, "1 question\nHigh bias", ha='center', fontsize=9, style='italic', color='gray')
            
        elif idx == 1:
            for i, (x, y) in enumerate([(5, 5.5), (2.5, 3.5), (7.5, 3.5)]):
                ax.text(x, y, "Is > X?", ha='center', va='center', fontsize=11, 
                        bbox=dict(boxstyle='round', facecolor='lightblue', edgecolor='black'))
            for x in [2.5, 7.5]:
                ax.annotate('', xy=(x, 4), xytext=(5, 5), arrowprops=dict(arrowstyle='->'))
            for x in [1, 4]:
                ax.text(x, 1.5, "Leaf", ha='center', va='center', fontsize=10,
                        bbox=dict(boxstyle='round', facecolor='lightgreen', edgecolor='black'))
            for x in [6, 9]:
                ax.text(x, 1.5, "Leaf", ha='center', va='center', fontsize=10,
                        bbox=dict(boxstyle='round', facecolor='lightgreen', edgecolor='black'))
            for x in [2.5, 7.5]:
                for lx in [x-1.5, x+1.5]:
                    ax.annotate('', xy=(lx, 2), xytext=(x, 3), arrowprops=dict(arrowstyle='->'))
            ax.text(5, 0.5, "~3 questions\nGood balance", ha='center', fontsize=9, style='italic', color='gray')
            
        else:
            for i, layer in enumerate([(5,), (3, 7), (1.5, 4.5, 6.5, 9.5)]):
                y = 5.5 - i * 1.2
                for x in layer:
                    ax.text(x, y, "?", ha='center', va='center', fontsize=10,
                            bbox=dict(boxstyle='round', facecolor='lightblue', edgecolor='black'))
            for i, layer in enumerate([(3, 7), (1.5, 4.5, 6.5, 9.5)]):
                y_from = 5.5 - (i) * 1.2
                y_to = 5.5 - (i+1) * 1.2
                for x in layer:
                    ax.annotate('', xy=(x, y_to+0.3), xytext=(5 if i==0 else layer[0]-1.5, y_from-0.3), 
                                arrowprops=dict(arrowstyle='->', lw=0.5))
            ax.text(5, 0.5, "Many questions\nHigh variance", ha='center', fontsize=9, style='italic', color='gray')
    
    plt.tight_layout()
    plt.show()

draw_simple_vs_complex()

## Summary

Decision trees work by:
1. **Starting at the root** - asking the most important question first
2. **Splitting recursively** - each answer leads to the next question
3. **Stopping at leaves** - when a decision is reached

The depth of the tree controls the tradeoff:
- **Shallow trees** (underfit): May miss important patterns
- **Deep trees** (overfit): May memorize noise in training data
- **Optimal depth**: Balances bias and variance

In the Ames Housing notebook (Part 5), you'll see this in action with real housing data!